# 05 Capability Review

Stage 5 converts selected evidence into evidence-backed capability themes.

These capability themes act as the bridge between individual accomplishments and resume-level positioning. They help determine what should appear in the summary, Core Expertise section, and experience narrative without drifting into unsupported keywords or generic branding.

The LLM groups evidence into candidate capabilities, while the human review step normalizes the names into candidate-focused language and flags any interpretation risks.


Inputs:
- `artifacts/selected_evidence.json`
- `artifacts/target_archetype.json`

Output:
- `artifacts/capability_review.json`

This stage uses LLM judgment, but only after Stage 4 has selected and scored the underlying evidence.



## 5B. Prepare Evidence for Capability Extraction

Use must_include, strong_include, and optional evidence.
Compress-tier evidence can support context but should not dominate.

## 5C. Generate Capability Review

Infer demonstrated capabilities from selected evidence.

## 5D. Validate Capability Review

Check:
- each capability has evidence_ids
- each evidence_id exists
- no capability is unsupported
- no production claim is based only on teaching evidence

## 5E. Save capability_review.json

Suggested output shape
```json
{
  "capability_review_summary": {
    "target_archetype": "Principal AI Architect / AI Platform Engineering Lead",
    "source_artifact": "selected_evidence.json",
    "capability_count": 0,
    "method": "Capabilities inferred from selected evidence and target archetype signals."
  },
  "capabilities": [
    {
      "capability": "AI Platform Architecture",
      "strength": 5,
      "evidence_ids": [
        "exp_01_01_01_04",
        "exp_01_01_02_01"
      ],
      "evidence_sources": [
        "Consolidated Edison",
        "NBCUniversal"
      ],
      "supporting_evidence_summary": "Supported by governed GenAI analytics assistant work and AWS-based AI/graph analytics pipeline architecture.",
      "target_relevance": "High",
      "resume_use": "core_expertise",
      "cautions": [
        "Avoid overstating NBCUniversal work as long-term production ownership."
      ]
    }
  ]
}
```

## 5A. Load Selected Evidence

Inputs:
- selected_evidence.json
- target_archetype.json

In [17]:
%run ./init_notebook.py


Repo root: /Users/douglasdaly/GitHub/Generative-AI
Added src to sys.path: /Users/douglasdaly/GitHub/Generative-AI/src
Resume builder notebooks: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder
Artifacts: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder/artifacts


In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from genai_demos.resume_builder.config import ARTIFACT_DIR
from genai_demos.resume_builder.helpers import load_json, save_json
from genai_demos.resume_builder.capabilities import call_json_model
import json

load_dotenv()

MODEL = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)


In [3]:
selected_evidence = load_json(ARTIFACT_DIR / "selected_evidence.json")
target_archetype = load_json(ARTIFACT_DIR / "target_archetype.json")

selected_evidence.keys(), target_archetype.keys()

(dict_keys(['selection_summary', 'selected_evidence', 'excluded_evidence']),
 dict_keys(['title', 'archetype_summary', 'market_basis', 'hypothesis_assessment', 'core_market_themes', 'target_capabilities', 'target_problem_spaces', 'target_technology_areas', 'seniority_expectations', 'success_patterns', 'resume_implications', 'archetype_narrative']))

In [4]:
selected_evidence["selection_summary"]

{'target_archetype': 'Principal AI Architect / AI Platform Engineering Lead',
 'total_evidence_items': 44,
 'selected_count': 35,
 'excluded_count': 9,
 'selection_method': 'Deterministic first-pass selection using field_capped_score, direct_match_count, rank, and qualitative caution flags. This step selects evidence, not final resume bullets.',
 'tier_counts': {'must_include': 6,
  'strong_include': 9,
  'optional': 9,
  'compress': 11,
  'exclude': 9}}

## 5B. Prepare Evidence for Capability Review

In [5]:
CAPABILITY_SOURCE_TIERS = {"must_include", "strong_include", "optional"}

capability_source_evidence = [
    item
    for item in selected_evidence["selected_evidence"]
    if item["selection_tier"] in CAPABILITY_SOURCE_TIERS
]

len(capability_source_evidence)

24

In [6]:
def compact_evidence_for_capability_review(items):
    return [
        {
            "evidence_id": item["evidence_id"],
            "selection_tier": item["selection_tier"],
            "source_label": item["source_label"],
            "evidence_text": item["evidence_text"],
            "primary_supporting_signals": item.get("primary_supporting_signals", []),
            "cautions": item.get("cautions", []),
        }
        for item in items
    ]

capability_evidence = compact_evidence_for_capability_review(capability_source_evidence)

len(capability_evidence), capability_evidence[:2]

(24,
 [{'evidence_id': 'exp_01_01_01_01',
   'selection_tier': 'must_include',
   'source_label': 'Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)',
   'evidence_text': 'Built a governed analytics assistant using Gemini, BigQuery, and tool-calling architectures that applied analyst rules, operational controls, and auditable execution patterns for AI-enabled construction and financial analytics.',
   'primary_supporting_signals': ['Productionization and operationalization of generative and agentic AI',
    'Governance, compliance, and responsible AI as non-negotiable requirements',
    'Evaluating, integrating, and operationalizing emerging AI technologies (GenAI, agentic AI, RAG)',
    'Enterprise-scale AI/ML platform architecture and delivery',
    'Architecting and designing scalable AI/ML platforms and solutions'],
   'cautions': []},
  {'evidence_id': 'exp_01_08_01_03',
   'selection_tier': 'must_include',
   'source_label': 'Capital One | Senio

## 5C. Build Prompt

In [7]:
def build_capability_review_prompt(capability_evidence, target_archetype):
    expected = {
        "capability_review_summary": {
            "target_archetype": target_archetype.get("title", ""),
            "source_artifact": "selected_evidence.json",
            "capability_count": 0,
            "method": "Capabilities inferred from selected evidence and target archetype."
        },
        "capabilities": [
            {
                "capability": "string",
                "strength": 0,
                "evidence_ids": ["string"],
                "evidence_sources": ["string"],
                "supporting_evidence_summary": "string",
                "target_relevance": "high | medium | low",
                "resume_use": "core_expertise | summary | experience | supporting_context",
                "cautions": ["string"]
            }
        ]
    }

    return f"""
You are reviewing selected resume evidence to identify demonstrated capabilities.

This is capability discovery, not resume writing.

Target archetype:
{target_archetype.get("title", "")}

Archetype summary:
{target_archetype.get("archetype_summary", "")}

Selected evidence:
{json.dumps(capability_evidence, indent=2)}

Task:
Infer the demonstrated capabilities supported by the selected evidence.

Rules:
- Use only the provided selected evidence.
- Every capability must have at least one supporting evidence_id.
- Do not create capabilities that are not supported by evidence.
- Do not turn every keyword into a capability.
- Prefer durable capability themes over tool lists.
- Do not write resume bullets.
- Do not exaggerate teaching evidence as production ownership.
- Preserve cautions when they affect how the capability should be used.
- Keep only capabilities with strength >= 3.
- Return 8 to 14 capabilities.
- Use role_context and summary_context to understand why evidence mattered.
- Do not create a capability from context alone.
- A capability must be supported by at least one evidence_text item.
- Context may clarify scale, business importance, operating environment, and scope.
- Preserve cautions when context increases apparent scope but the evidence_text is narrow.

Strength scale:
5 = strongly demonstrated across multiple high-quality evidence items
4 = clearly demonstrated by one strong item or several moderate items
3 = credible but narrower or less central
2 = adjacent support only, exclude
1 = weak support, exclude

Resume use values:
- core_expertise
- summary
- experience
- supporting_context

Return valid JSON only.

Return exactly this structure:
{json.dumps(expected, indent=2)}
"""

## 5D. Generate Capability Review

In [8]:
prompt = build_capability_review_prompt(
    capability_evidence=capability_evidence,
    target_archetype=target_archetype,
)

capability_review = call_json_model(prompt, MODEL)

capability_review.keys()

dict_keys(['capability_review_summary', 'capabilities'])

## 5E. Validate

In [9]:
valid_evidence_ids = {item["evidence_id"] for item in capability_source_evidence}

def validate_capability_review(capability_review, valid_evidence_ids):
    errors = []

    for idx, cap in enumerate(capability_review.get("capabilities", [])):
        if not cap.get("capability"):
            errors.append(f"capabilities[{idx}] missing capability")

        strength = cap.get("strength")
        if not isinstance(strength, int) or not (3 <= strength <= 5):
            errors.append(f"capabilities[{idx}] invalid strength: {strength}")

        evidence_ids = cap.get("evidence_ids", [])
        if not evidence_ids:
            errors.append(f"capabilities[{idx}] has no evidence_ids")

        for evidence_id in evidence_ids:
            if evidence_id not in valid_evidence_ids:
                errors.append(
                    f"capabilities[{idx}] unknown evidence_id: {evidence_id}"
                )

    return errors

capability_errors = validate_capability_review(
    capability_review,
    valid_evidence_ids,
)

capability_errors

[]

## 5F. Save Output

In [10]:
save_json(capability_review, ARTIFACT_DIR / "capability_review.json")

In [11]:
for cap in capability_review["capabilities"]:
    print("\n" + "=" * 100)
    print(cap["capability"], "| strength:", cap["strength"], "| use:", cap["resume_use"])
    print("Relevance:", cap["target_relevance"])
    print("Evidence:", ", ".join(cap["evidence_ids"]))
    print(cap["supporting_evidence_summary"])

    if cap.get("cautions"):
        print("Cautions:")
        for caution in cap["cautions"]:
            print(" -", caution)


Productionization and operationalization of generative and agentic AI | strength: 5 | use: core_expertise
Relevance: high
Evidence: exp_01_01_01_01, exp_01_02_01_02
Demonstrated by building a governed analytics assistant using Gemini, BigQuery, and tool-calling architectures with operational controls and auditable execution patterns, and by teaching practical AI development workflows including generative and agentic AI.
Cautions:
 - Teaching evidence supports mentorship, technical fluency, and developer enablement; do not present it as production ownership unless the bullet explicitly supports that.

Enterprise-scale AI/ML platform architecture and delivery | strength: 5 | use: core_expertise
Relevance: high
Evidence: exp_01_01_01_01, exp_01_01_02_01
Built governed analytics assistants and AWS-based AI and graph analytics pipelines using cloud-native tools, demonstrating architecture and delivery of scalable AI/ML platforms.
Cautions:
 - Frame carefully as short-cycle architecture/pro

## 5G. Manually Patch Capability Names to Candidate-Focused Terms

The LLM groups selected evidence into capability buckets. This section applies a human naming pass so the capabilities read like the candidate's demonstrated strengths rather than copied market-signal language.

This is intentional human-in-the-loop review.

In [12]:
CAPABILITY_RENAMES = {
    "Enterprise-scale AI/ML platform architecture and delivery":
        "Enterprise AI and Operational Intelligence Platforms",

    "Productionization and operationalization of generative and agentic AI":
        "Governed GenAI and Tool-Calling Systems",

    "Governance, compliance, and responsible AI practices":
        "AI Governance, Controls, and Responsible Adoption",

    "MLOps, LLMOps, and operational observability":
        "Observability, Monitoring, and Operational Controls",

    "Technical leadership, mentorship, and cross-functional influence":
        "Technical Leadership, Mentorship, and Talent Development",

    "Cloud-native, scalable, and resilient system design":
        "Cloud-Native AI Infrastructure and Data Platforms",

    "Translating business requirements into technical solutions":
        "Business Translation and Decision Support",

    "Developing reusable frameworks, components, and developer tools":
        "Reusable Frameworks and Developer Enablement",

    "Cross-functional collaboration and stakeholder alignment":
        "Cross-Functional Strategy and Executive Alignment",

    "Strategic and visionary leadership (roadmap, technical vision, innovation)":
        "Technical Roadmaps, Strategy, and Innovation",

    "AI/ML model lifecycle management and deployment":
        "AI/ML Deployment and Lifecycle Support",

    "Decision support and advanced analytics":
        "Advanced Analytics and Decision Support",
}

In [13]:
def apply_capability_name_patches(capability_review, rename_map):
    """
    HUMAN_REVIEW PATCH.

    Applies candidate-focused naming to LLM-generated capability buckets.

    The LLM is responsible for semantic grouping. This function is a deliberate
    human-in-the-loop naming pass so capabilities reflect the candidate's actual
    positioning and writing style.

    This is not a STUB. It is an intentional review layer.
    """
    for cap in capability_review.get("capabilities", []):
        original = cap["capability"]
        cap["original_capability"] = original
        cap["capability"] = rename_map.get(original, original)

    return capability_review


capability_review = apply_capability_name_patches(
    capability_review,
    CAPABILITY_RENAMES,
)

In [14]:
# Re-weight capabilities based on personal judgment
for cap in capability_review["capabilities"]:
    if cap["capability"] == "AI Governance, Controls, and Responsible Adoption":
        cap["strength"] = 4

In [15]:
import pandas as pd
pd.options.display.max_colwidth=0
def build_capability_summary_table(capability_review):
    rows = []

    for cap in capability_review.get("capabilities", []):
        rows.append({
            "capability": cap.get("capability"),
            "original_capability": cap.get("original_capability", cap.get("capability")),
            "strength": cap.get("strength"),
            "target_relevance": cap.get("target_relevance"),
            "resume_use": cap.get("resume_use"),
            "evidence_count": len(cap.get("evidence_ids", [])),
            "evidence_sources": ", ".join(cap.get("evidence_sources", [])),
            "caution_count": len(cap.get("cautions", [])),
        })

    return pd.DataFrame(rows).sort_values(
        by=["strength", "target_relevance", "evidence_count"],
        ascending=[False, True, False],
    )


capability_summary_df = build_capability_summary_table(capability_review)
capability_summary_df

,capability,original_capability,strength,target_relevance,resume_use,evidence_count,evidence_sources,caution_count
2,"MLOps, LLMOps, and observability as operational foundations","MLOps, LLMOps, and observability as operational foundations",5,high,core_expertise,3,"Capital One | Senior Manager, Operations Analysis, Meta | Data Scientist",1
4,Reusable Frameworks and Developer Enablement,"Developing reusable frameworks, components, and developer tools",5,high,core_expertise,3,"Meta | Data Scientist, Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting), Nike (via Intersoft Inc.) | Expert SEO Solutions Architect",0
5,"Mentorship, cross-functional leadership, and technical influence","Mentorship, cross-functional leadership, and technical influence",5,high,core_expertise,3,"SimpliLearn & Interview Kickstart | AI Instructor, Raytheon | Senior Principal Systems Engineer / Program Manager",2
0,Governed GenAI and Tool-Calling Systems,Productionization and operationalization of generative and agentic AI,5,high,core_expertise,2,"Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting), SimpliLearn & Interview Kickstart | AI Instructor",1
1,Enterprise AI and Operational Intelligence Platforms,Enterprise-scale AI/ML platform architecture and delivery,5,high,core_expertise,2,"Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting), Daly Engineers LLC | Principal Consultant | NBCUniversal (via Apex Systems)",1
6,Business Translation and Decision Support,Translating business requirements into technical solutions,4,high,summary,3,"Capital One | Senior Manager, Operations Analysis, Nike (via Intersoft Inc.) | Expert SEO Solutions Architect, Meta | Data Scientist",0
8,"Technical Roadmaps, Strategy, and Innovation","Strategic and visionary leadership (roadmap, technical vision, innovation)",4,high,summary,3,"Capital One | Senior Manager, Operations Analysis, Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)",2
3,"Governance, compliance, and responsible AI","Governance, compliance, and responsible AI",4,high,summary,2,"Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting), SimpliLearn & Interview Kickstart | AI Instructor",1
7,Cloud-Native AI Infrastructure and Data Platforms,"Cloud-native, scalable, and resilient system design",4,high,summary,2,"Daly Engineers LLC | Principal Consultant | NBCUniversal (via Apex Systems), Capital One | Senior Manager, Operations Analysis",2
10,"Business value delivery (growth, efficiency, customer impact)","Business value delivery (growth, efficiency, customer impact)",4,high,summary,2,"Samsung Electronics (via Harvey Nash) | Data Scientist, Raytheon | Senior Principal Systems Engineer / Program Manager",1


In [16]:
save_json(capability_review, ARTIFACT_DIR / "capability_review.json")

capability_summary_df.to_csv(
    ARTIFACT_DIR / "capability_review_summary.csv",
    index=False,
)